In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\adeyi\Downloads\HealthConnect_Appointment_Data.csv")
print (f"Rows:{df.shape[0]}, Columns:{df.shape[1]}")
df.head()

Rows:5000, Columns:18


,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,29.0,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,2,6,0,Yes,SMS,14.3,42.0,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,38,5,1,Yes,SMS,11.4,11.0,No-Show
3,HC-00004,P-1031,Male,59,55-64,Follow-up,7/18/2025,8/28/2025,Thursday,Evening,41,3,1,Yes,SMS,7.4,35.0,Attended
4,HC-00005,P-1458,Female,34,25-34,Follow-up,7/9/2025,8/25/2025,Monday,Afternoon,47,3,1,Yes,Email,5.6,27.0,No-Show


In [2]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   appointment_id         5000 non-null   object 
 1   patient_id             5000 non-null   object 
 2   gender                 5000 non-null   object 
 3   age                    5000 non-null   int64  
 4   age_group              5000 non-null   object 
 5   appointment_type       5000 non-null   object 
 6   booking_date           5000 non-null   object 
 7   appointment_date       5000 non-null   object 
 8   appointment_day        5000 non-null   object 
 9   appointment_time       5000 non-null   object 
 10  booking_lead_days      5000 non-null   int64  
 11  previous_appointments  5000 non-null   int64  
 12  previous_no_shows      5000 non-null   int64  
 13  reminder_sent          5000 non-null   object 
 14  reminder_channel       3634 non-null   object 
 15  dist

In [3]:
missing = df.isnull().sum()
missing_pct = (missing/len(df) * 100).round(2)
pd.DataFrame({'Missing_count': missing, 'missing_pct': missing_pct})[missing > 0].sort_values('missing_pct', ascending=False)


,Missing_count,missing_pct
reminder_channel,1366,27.32
distance_to_clinic_km,90,1.80
waiting_time_minutes,60,1.20


In [4]:
#Observations on Missing Data
#Looking at the missing values above, there are three columns with gaps:

#reminder_channel is missing for 1,366 rows. This is not a mistake in the data, it lines up exactly with the rows where reminder_sent is "No." In other words, if no reminder was sent, there's naturally no channel to record. This should be treated as "Not Applicable" rather than a data problem.

#distance_to_clinic_km is missing for 90 rows (about 1.8% of the data).

#waiting_time_minutes is missing for 60 rows (about 1.2% of the data).
#These last two are small enough that they likely don't affect the overall analysis much, but I will decide later (in Week 5) whether to exclude these rows or fill in the missing values before calculating KPIs.

In [5]:
print("Duplicate Rows:", df.duplicated().sum())
print("Duplicate appointment_id:", df['appointment_id'].duplicated().sum())


Duplicate Rows: 0
Duplicate appointment_id: 0


In [6]:
df['booking_date_dt'] = pd.to_datetime(df['booking_date'], format='%m/%d/%Y')
df['appointment_date_dt'] = pd.to_datetime(df['appointment_date'], format='%m/%d/%Y')
df['calc_lead_days'] = (df['appointment_date_dt'] - df['booking_date_dt']).dt.days

checks = {
    'lead days match': (df['calc_lead_days'] == df['booking_lead_days']).all(),
    'no negative lead days': (df['calc_lead_days'] >= 0).all(),
    'weekday matches appointment_day': (df['appointment_date_dt'].dt.day_name() == df['appointment_day']).all(),
    'no_shows <= previous_appointments': (df['previous_no_shows'] <= df['previous_appointments']).all(),
}
for check, passed in checks.items():
    print(("PASS" if passed else "FAIL"), "-", check)

PASS - lead days match
PASS - no negative lead days
PASS - weekday matches appointment_day
PASS - no_shows <= previous_appointments


In [7]:
outcome_counts = df['appointment_outcome'].value_counts()
outcome_pct = df['appointment_outcome'].value_counts(normalize=True).mul(100).round(1)
pd.DataFrame({'count': outcome_counts, 'pct': outcome_pct})

,count,pct
appointment_outcome,,
No-Show,2423,48.5
Attended,2314,46.3
Cancelled,263,5.3


In [8]:
pd.crosstab(df['reminder_sent'], df['appointment_outcome'], normalize='index').mul(100).round(1)

appointment_outcome,Attended,Cancelled,No-Show
reminder_sent,,,
No,42.7,5.9,51.4
Yes,47.6,5.0,47.4
